In [ ]:
import aplpy
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
import glob
import os

In [ ]:
def examinar_headers(campo):
    """
    Examina en detalle la estructura de los archivos FITS
    """
    base_path = f"../anac_data/{campo}"
    
    archivos = [
        f"{base_path}/{campo}_F861.fits.fz",
        f"{base_path}/{campo}_F660.fits.fz", 
        f"{base_path}/{campo}_F515.fits.fz"
    ]
    
    for archivo in archivos:
        print(f"\n{'='*80}")
        print(f"EXAMINANDO: {os.path.basename(archivo)}")
        print(f"{'='*80}")
        
        if not os.path.exists(archivo):
            print("❌ Archivo no existe")
            continue
            
        try:
            with fits.open(archivo) as hdul:
                print(f"Número de HDUs: {len(hdul)}")
                
                for i, hdu in enumerate(hdul):
                    print(f"\n--- HDU {i} ---")
                    print(f"Tipo: {type(hdu)}")
                    print(f"Forma: {hdu.data.shape if hdu.data is not None else 'Sin datos'}")
                    print(f"Dimensión: {hdu.header.get('NAXIS', 'N/A')}")
                    
                    # Información clave del header
                    keys_importantes = [
                        'SIMPLE', 'BITPIX', 'NAXIS', 'NAXIS1', 'NAXIS2', 'NAXIS3',
                        'EXTEND', 'BSCALE', 'BZERO', 'ORIGIN', 'TELESCOP', 'INSTRUME',
                        'OBJECT', 'RA', 'DEC', 'EQUINOX', 'RADESYS', 'CTYPE1', 'CTYPE2',
                        'CRPIX1', 'CRPIX2', 'CRVAL1', 'CRVAL2', 'CD1_1', 'CD1_2', 
                        'CD2_1', 'CD2_2', 'BUNIT', 'EXPTIME', 'FILTER', 'DATE-OBS',
                        'MJD-OBS', 'AIRMASS'
                    ]
                    
                    print("Header keys importantes:")
                    for key in keys_importantes:
                        if key in hdu.header:
                            print(f"  {key}: {hdu.header[key]}")
                    
                    # Mostrar las primeras 10 keys
                    print("\nPrimeras 10 keys del header:")
                    for j, key in enumerate(list(hdu.header.keys())[:10]):
                        print(f"  {key}: {hdu.header[key]}")
                        
        except Exception as e:
            print(f"❌ Error al leer archivo: {e}")

In [ ]:
def crear_imagen_aplpy_robusta(campo, output_name=None):
    """
    Versión corregida que maneja adecuadamente archivos FITS comprimidos
    """
    base_path = f"../anac_data/{campo}"
    
    # Crear directorio temporal
    temp_dir = f"{base_path}/temp"
    os.makedirs(temp_dir, exist_ok=True)
    
    # Archivos temporales descomprimidos
    temp_r = f"{temp_dir}/{campo}_F861.fits"
    temp_g = f"{temp_dir}/{campo}_F660.fits"
    temp_b = f"{temp_dir}/{campo}_F515.fits"
    
    # Archivos de proceso
    rgb_cube = f"{temp_dir}/{campo}_rgb_cube.fits"
    rgb_png = f"{temp_dir}/{campo}_rgb_image.png"
    
    print(f"Procesando {campo} desde archivos comprimidos...")
    
    try:
        # 1. Extraer datos de las CompImageHDU (HDU 1)
        print(" - Descomprimiendo HDUs...")
        
        def extraer_hdu_comprimida(archivo_entrada, archivo_salida):
            with fits.open(archivo_entrada) as hdul:
                # HDU 1 contiene los datos comprimidos
                comp_hdu = hdul[1]
                data = comp_hdu.data
                header = comp_hdu.header
                
                # Crear un PrimaryHDU estándar con los datos
                primary_hdu = fits.PrimaryHDU(data=data, header=header)
                primary_hdu.writeto(archivo_salida, overwrite=True)
                print(f"   ✓ {os.path.basename(archivo_entrada)} → {data.shape}")
        
        extraer_hdu_comprimida(f"{base_path}/{campo}_F861.fits.fz", temp_r)
        extraer_hdu_comprimida(f"{base_path}/{campo}_F660.fits.fz", temp_g)
        extraer_hdu_comprimida(f"{base_path}/{campo}_F515.fits.fz", temp_b)
        
        # 2. Crear cubo RGB
        print(" - Creando cubo RGB...")
        aplpy.make_rgb_cube([temp_r, temp_g, temp_b], rgb_cube)
        
        # 3. Crear imagen RGB con ajustes optimizados
        print(" - Generando imagen RGB...")
        aplpy.make_rgb_image(
            rgb_cube, rgb_png,
            stretch_r='log', stretch_g='log', stretch_b='log',
            vmin_r=0.01, vmin_g=0.01, vmin_b=0.01,
            vmax_r=0.95, vmax_g=0.95, vmax_b=0.95
        )
        
        # 4. Mostrar resultado con información del header
        print(" - Cargando y mostrando resultado...")
        img = plt.imread(rgb_png)
        fig, ax = plt.subplots(figsize=(16, 14), facecolor='black')
        ax.imshow(img)
        ax.axis('off')
        
        # Información detallada del campo
        with fits.open(temp_r) as hdul:
            header = hdul[0].header
            ra = header.get('CRVAL1', 'N/A')
            dec = header.get('CRVAL2', 'N/A')
            filter_r = header.get('FILTER', 'N/A')
            exptime_r = header.get('EXPTIME', 'N/A')
            mjd_obs = header.get('MJD-OBS', 'N/A')
        
        with fits.open(temp_g) as hdul:
            header_g = hdul[0].header
            filter_g = header_g.get('FILTER', 'N/A')
            exptime_g = header_g.get('EXPTIME', 'N/A')
        
        with fits.open(temp_b) as hdul:
            header_b = hdul[0].header
            filter_b = header_b.get('FILTER', 'N/A')
            exptime_b = header_b.get('EXPTIME', 'N/A')
        
        # Título principal
        plt.title(f'Centaurus A - {campo}\nTelescope: T80/T80Cam - J-PLUS Survey', 
                  color='white', size=18, weight='bold', pad=30)
        
        # Información detallada
        info_text = f'''Coordinates: RA={ra:.6f}°, DEC={dec:.6f}°
Filters: {filter_r}(R), {filter_g}(G), {filter_b}(B)
Exposure: {exptime_r}s(R), {exptime_g}s(G), {exptime_b}s(B)
MJD: {mjd_obs}
Pixel Scale: 0.55"/pixel • FOV: ~1.7°×1.7°
Image Size: 11000×11000 pixels'''

        ax.text(0.02, 0.98, info_text, transform=ax.transAxes, color='white', 
                fontsize=12, verticalalignment='top', family='monospace',
                bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
        
        # Barra de escala aproximada (basada en el pixel scale)
        height = img.shape[0]
        # 10 arcmin ≈ 1090 pixels (0.55"/pixel)
        scale_pixels = int(600 / 0.55)  # 10 arcmin en pixels
        ax.plot([50, 50 + scale_pixels], [100, 100], 
                color='yellow', linewidth=6)
        ax.text(50 + scale_pixels/2, 80, '10 arcmin', 
                color='yellow', ha='center', va='top', 
                fontsize=14, weight='bold')
        
        if output_name:
            plt.savefig(output_name, dpi=300, bbox_inches='tight', 
                       facecolor='black', edgecolor='none')
            print(f"✅ Imagen guardada: {output_name}")
        
        plt.show()
        
        # 5. Limpiar archivos temporales
        print(" - Limpiando archivos temporales...")
        for temp_file in [temp_r, temp_g, temp_b, rgb_cube, rgb_png]:
            if os.path.exists(temp_file):
                os.remove(temp_file)
        
        # Limpiar directorio temp si está vacío
        try:
            os.rmdir(temp_dir)
        except:
            pass
            
        print(" ✅ ¡Procesamiento completado exitosamente!")
        return fig
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return None


In [ ]:
# Examinar los headers
examinar_headers('CenA01')

In [ ]:
# Probar versión robusta
fig_cena01_robusto = crear_imagen_aplpy_robusta('CenA01', '../anac_data/Figs-images/CenA01_rgb_robusto.png')